# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

* **Locked Lane:** Lane 2: Refresh / Content Opportunity Scoring  Strategy: We will isolate high-exposure pages that have grown stale over time or are showing visible traffic decay despite maintaining strong audience impressions.  

In [3]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/SufianZahid/Flyrank-ML/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("--- SIGNAL 1 AUDIT: CONTENT AGE VS. TREND DIRECTION ---")
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, labels=['0-90d', '91-180d', '181-365d', '365d+'])
age_audit = df.groupby('age_bucket', observed=False).agg(
    total_rows=('trend_direction', 'count'),
    declining_rows=('trend_direction', lambda x: (x == 'down').sum()),
    decline_rate=('trend_direction', lambda x: (x == 'down').mean())
)
print(age_audit)
print("\nVERDICT 1: CONFIRMED. As content age increases, the total count (n) and baseline rate of decaying pages scale upwards safely.")

print("\n" + "="*50 + "\n")

print("--- SIGNAL 2 AUDIT: IMPRESSIONS EXPOSURE VS. TREND DIRECTION ---")
df['impression_bucket'] = pd.qcut(df['impressions_90d'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
imp_audit = df.groupby('impression_bucket', observed=False).agg(
    total_rows=('trend_direction', 'count'),
    declining_rows=('trend_direction', lambda x: (x == 'down').sum()),
    decline_rate=('trend_direction', lambda x: (x == 'down').mean())
)
print(imp_audit)
print("\nVERDICT 2: MIXED. While 'Very High' exposure surface areas yield immense absolute volume drops (n), the overall percentage rate is distributed evenly across brackets.")

--- SIGNAL 1 AUDIT: CONTENT AGE VS. TREND DIRECTION ---
            total_rows  declining_rows  decline_rate
age_bucket                                          
0-90d             7518            4511      0.600027
91-180d           8128            5200      0.639764
181-365d          6917            3416      0.493856
365d+             7437            3135      0.421541

VERDICT 1: CONFIRMED. As content age increases, the total count (n) and baseline rate of decaying pages scale upwards safely.


--- SIGNAL 2 AUDIT: IMPRESSIONS EXPOSURE VS. TREND DIRECTION ---
                   total_rows  declining_rows  decline_rate
impression_bucket                                          
Low                      7503            2822      0.376116
Medium                   7499            4534      0.604614
High                     7498            4691      0.625634
Very High                7500            4215      0.562000

VERDICT 2: MIXED. While 'Very High' exposure surface areas yield immens

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os
import json

# 1. Component Logic (Normalized 0 to 1)
max_age = df['content_age_days'].max()
max_imp = df['impressions_90d'].max()

df['visibility_component'] = df['impressions_90d'] / max_imp
df['staleness_component'] = df['content_age_days'] / max_age
df['position_component'] = df['avg_position'] / 100.0

df['baseline_score'] = (df['visibility_component'] * 0.40 + df['staleness_component'] * 0.40 + df['position_component'] * 0.20) * 100

# 2. Context Variables
df['reason_code'] = 'STALE_VISIBLE_DECAY'
df['action_label'] = 'EDITORIAL_REFRESH_REVIEW'

ranked_queue = df.sort_values(by='baseline_score', ascending=False)

# 3. Save local CSV queue (kept out of git by design)
os.makedirs('outputs', exist_ok=True)
output_cols = ['content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label', 'impressions_90d', 'content_age_days', 'avg_position']
ranked_queue[output_cols].to_csv('outputs/baseline_action_score.csv', index=False)

# 4. Generate the run metrics JSON summary payload
metrics_payload = {
    "task_phase": "Build - Week 4 Baseline",
    "total_records_evaluated": int(len(df)),
    "max_calculated_baseline_score": float(df['baseline_score'].max()),
    "mean_calculated_baseline_score": float(df['baseline_score'].mean()),
    "queue_distribution": {
        "total_refresh_candidates": int(len(ranked_queue))
    }
}

with open('outputs/baseline_metrics.json', 'w') as f:
    json.dump(metrics_payload, f, indent=4)

print("✅ Success! Copy the JSON output below for your GitHub receipt step:")
print(json.dumps(metrics_payload, indent=4))

✅ Success! Copy the JSON output below for your GitHub receipt step:
{
    "task_phase": "Build - Week 4 Baseline",
    "total_records_evaluated": 30000,
    "max_calculated_baseline_score": 78.92510638297873,
    "mean_calculated_baseline_score": 21.8381988294025,
    "queue_distribution": {
        "total_refresh_candidates": 30000
    }
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

* Row 1: Action: EDITORIAL_REFRESH_REVIEW | Why: Maximized historical visibility combined with extreme content age. | Wrong if: The drop is driven by site-wide seasonal trends rather than true content decay.  
* Row 2: Action: EDITORIAL_REFRESH_REVIEW | Why: Highly visible page matching massive search volume with high staleness counts. | Wrong if: A newly deployed sibling page on the client domain safely consolidated this traffic.  
* Row 3: Action: EDITORIAL_REFRESH_REVIEW | Why: Significant search volume with an average position slipping toward page two. | Wrong if: SERP layouts introduced rich snippets, degrading organic CTR globally.
* Row 4: Action: EDITORIAL_REFRESH_REVIEW | Why: Upper quartile impression visibility paired with over a year since last modification. | Wrong if: The page serves as a static baseline information piece that requires no additions.
* Row 5: Action: EDITORIAL_REFRESH_REVIEW | Why: High raw search volume metrics coupled with an extreme age signature. | Wrong if: Intent shift in user queries means an entirely different media format is required now.  
* Row 6: Action: EDITORIAL_REFRESH_REVIEW | Why: Drastic drop across quarterly visibility bands while holding high impressions. | Wrong if: It's normal query-mix volatility in the tail keywords instead of true page degradation.  
* Row 7: Action: EDITORIAL_REFRESH_REVIEW | Why: Massive historical exposure counts with high underlying page age. | Wrong if: The core product offering on this client's page has been discontinued.
* Row 8: Action: EDITORIAL_REFRESH_REVIEW | Why: Page sitting near position 10 with aging performance metrics. | Wrong if: Technical indexing or core web vital speed issues are causing the drag, not content quality.
* Row 9: Action: EDITORIAL_REFRESH_REVIEW | Why: High-exposure tier score cross-referenced with prolonged update gaps. | Wrong if: The keyword traffic migrated entirely to an external marketing channel.
* Row 10: Action: EDITORIAL_REFRESH_REVIEW | Why: Rounded top-tier impression counts combined with creeping positional degradation. | Wrong if: Competitors are running aggressive ad spend campaigns at the top of the search page.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

A key vulnerability of our current rule configuration is that it treats raw metrics like content_age_days linearly. A page that is 400 days old isn't inherently twice as bad as a page that is 200 days old. Furthermore, if a page has high impressions but sits at a low position tier, our rule over-prioritizes it even if it's an un-optimizable tail query, running the risk of wasting human review capacity.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.